# EXP-012 (red-team): synthetic form → VLM extraction → verification → draft FHIR → deterministic scorer

## 1 Requirements and predeclared hypothesis
H0: single-pass VLM extraction on 12 clean synthetic forms is not perfect; the consistency+identity gate admits only correct fields (false admissions = 0) and the cross-patient FCO mismatch is caught 12/12. Synthetic only; Epic NOT contacted; EHR write PROHIBITED.

## 2 Input source, license, hashes
derived/forms12 on magicSTUDIObox (Synthea sample CSV sha 990d5206..., rendered with Pillow 12.3.0, SYNFORM root 36d12313...); public export evidence/redteam/lanes/EHR_qwen2.5vl_3b_forms12_PUBLIC.jsonl (booleans only).

## 3 Frozen settings
qwen2.5vl:3b via Ollama, temperature 0, seed 20261102, format=json.

## 4 Deterministic preprocessing
frontier_forms.fhir_draft normalization and validation (dates ISO, gender map, FCO case-id cross-check).

## 5 Model/harness execution
replayed from receipts; no inference here.

## 6 Analysis
per-field accuracy for extract / extract_fco / reread; admission confusion (TP/FP/FN/TN, FPR, FDR); FHIR draft equality vs gold draft.

## 7 Negative controls
cross-patient identity check; fixture validation failures in tests/test_fhir_draft.py.

## 8 Outputs
exp012_forms.json.

## 9 FCO/FCG dependency changes
adds EXECUTED edge PROPOSED_TRANSFORM(Synthea→form→OCR→FHIR) for 12 synthetic cases; Epic edge unchanged NOT_TESTED.

## 10 Claim ceiling
Notebook PASS verifies the local replay only. It never upgrades NOT_TESTED, BLOCKED or NEGATIVE states, and it never attributes a causal gain to a hash, leaf or edge: hashes bind bytes, controlled interventions bind effects. 12 clean rendered forms are not scanned/faxed documents; no OCR robustness claim.


In [1]:
import json, hashlib, sys
from pathlib import Path
import os
R = Path.cwd(); sys.path.insert(0, str(R / 'src')); sys.path.insert(0, str(R / 'scripts/redteam')); sys.path.insert(0, str(R / 'scripts'))
def sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def jsonl(p): return [json.loads(x) for x in Path(p).read_text().splitlines() if x.strip()]
OUT = Path(os.environ.get('REDTEAM_RUN_DIR', str(R / 'evidence/notebook_runs_redteam_v1'))) / 'outputs'; OUT.mkdir(parents=True, exist_ok=True)
def emit(name, obj):
    p = OUT / name; p.write_text(json.dumps(obj, sort_keys=True, indent=2) + '\n'); print('OUTPUT', name, sha(p)); return p
from frontier_forms.fhir_draft import FORM_FIELDS, confusion, draft_fhir
pub = sorted((R / 'evidence/redteam/lanes').glob('EHR_*_forms12_PUBLIC.jsonl'))
if not pub:
    res = {'status': 'NOT_TESTED', 'reason': 'legacy 12-form lane dropped by operator at finish request (GPU contention); superseded by SYNFORM2 lanes in EXP-014/015'}
    emit('exp012_forms.json', res); print(res)
else:
    rows = jsonl(pub[0]); meta = json.loads(pub[0].with_name(pub[0].name.replace('_PUBLIC.jsonl', '_META.json')).read_text())
    assert len({r['case_id'] for r in rows}) == len(rows) == 12
    agg = {'n_forms': len(rows), 'model': meta['model'], 'form_root': meta['form_root'], 'per_field': {}, 'cross_patient_abstain': sum(1 for r in rows if r['cross_patient_identity_check'] == 'ABSTAIN')}
    for f in FORM_FIELDS:
        agg['per_field'][f] = {k: sum(1 for r in rows if r[k][f]) for k in ('correct_extract', 'correct_fco', 'correct_reread', 'admitted')}
    tot = {k: sum(1 for r in rows for f in FORM_FIELDS if r[k][f]) for k in ('correct_extract', 'correct_fco', 'correct_reread', 'admitted')}
    adm = confusion({(r['case_id'], f): r['admitted'][f] for r in rows for f in FORM_FIELDS} and {f'{r["case_id"]}|{f}': r['admitted'][f] for r in rows for f in FORM_FIELDS},
                    {f'{r["case_id"]}|{f}': r['correct_extract'][f] for r in rows for f in FORM_FIELDS}) if False else None
    # field-level admission confusion across all 84 fields
    A = {f'{r["case_id"]}|{f}': r['admitted'][f] for r in rows for f in FORM_FIELDS}; C = {f'{r["case_id"]}|{f}': r['correct_extract'][f] for r in rows for f in FORM_FIELDS}
    tp = sum(1 for k in A if A[k] and C[k]); fp = sum(1 for k in A if A[k] and not C[k]); fn = sum(1 for k in A if not A[k] and C[k]); tn = sum(1 for k in A if not A[k] and not C[k])
    div = lambda a, b: a / b if b else None
    agg.update({'fields_total': len(A), 'totals': tot, 'admission_confusion': {'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn, 'FPR': div(fp, fp + tn), 'FDR': div(fp, tp + fp), 'FNR': div(fn, tp + fn), 'AUROC': 'NOT_COMPUTED_NO_CONTINUOUS_SCORES'},
                'false_admissions': fp, 'draft_patient_matches_gold': sum(1 for r in rows if r['draft_patient_matches_gold']), 'draft_encounter_matches_gold': sum(1 for r in rows if r['draft_encounter_matches_gold']),
                'tokens_extract_total': sum(r['tokens_extract'] for r in rows), 'tokens_challenge_total': sum(r['tokens_challenge'] for r in rows),
                'wall_extract_total': round(sum(r['wall_extract'] for r in rows), 2), 'wall_challenge_total': round(sum(r['wall_challenge'] for r in rows), 2),
                'epic': 'NOT_CONTACTED', 'ehr_write': 'PROHIBITED_NOT_ATTEMPTED'})
    emit('exp012_forms.json', agg); print(json.dumps(agg, indent=2))

OUTPUT exp012_forms.json 6551d11f77a5bfdaf89a2d59da4c78c8fd958f95a7ac38f3abf4c5fcda4e2fd4
{'status': 'NOT_TESTED', 'reason': 'legacy 12-form lane dropped by operator at finish request (GPU contention); superseded by SYNFORM2 lanes in EXP-014/015'}
